# **ADMET and Toxicity Screening of Hygiene-Pass Molecules**

This notebook performs the next filtering stage in the generative design pipeline by evaluating the ADMET and toxicity profiles of molecules that passed the medicinal chemistry hygiene screen. The workflow loads the hygiene-pass molecules, runs ADMET/toxicity predictions using either a real predictor such as ADMET-AI or a placeholder stub, and applies a set of CNS-oriented filtering criteria to prioritize compounds with more favorable developability properties.

Predictions are combined with the previous hygiene-screening context, allowing both activity-related and developability-related information to be reviewed together. Molecules are then filtered using rule-based gates based on properties such as intestinal absorption, permeability, BBB penetration proxy, hERG risk, DILI risk, and CYP3A4 inhibition.

The notebook saves raw prediction outputs, annotated datasets, the subset of molecules that pass the ADMET screen, and a concise summary of pass rates and rules used. This step helps prioritize compounds with a more balanced profile before moving to later stages such as docking, ranking, or final selection.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install rdkit

In [ ]:
#!pip -q install admet-ai


In [ ]:
# =========================
#  ADMET / TOX FILTER
# =========================
# What this notebook does:
# 1) Loads the hygiene-pass molecules from Step 1 (CSV preferred; SMI fallback)
# 2) Runs ADMET/Toxicity predictions (use ADMET-AI if available or use stub)
# 3) Applies CNS-oriented default gates
# 4) Saves results into .../samples/analyses/admet-<timestamp>/ as step2_* files

from pathlib import Path
from datetime import datetime
import pandas as pd
import os
#from admet_ai import ADMETModel

# ---- choose target  ----
targets = ["5-HT6","ache","bace1","buche","esr1","3beta","mao-b"]
target  = targets[6]  # e.g., "5-HT6"

#BASE = Path(f"/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/reinvent/{target}/samples/analyses")

# ---- choose input  ----
PREFER_PASS_CSV = True       # True: use admet_hygiene_pass.csv; False -> try SMI first


# **1) Identify and Load Molecules that Passed the Medicinal Chemistry Hygiene Filter**

In [ ]:

# Base folders for the  workflow
FILTER_BASE = Path("/content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering")
HYG_BASE    = FILTER_BASE / "2-medchem_hygiene"
ADMET_BASE  = FILTER_BASE / "3-admet_pass"

PREFER_PASS_CSV = True

# --- Find the latest hygiene run for this target (or set explicitly) ---
hyg_runs = sorted(HYG_BASE.glob(f"{target}_hygiene-*"))
if not hyg_runs:
    raise FileNotFoundError(f"No {target}_hygiene-* folders found under: {HYG_BASE}")

HYG_DIR = hyg_runs[-1]
print("Using hygiene run:", HYG_DIR)


# --- Inputs produced by Step 2 (MedChem hygiene) ---
PASS_CSV = HYG_DIR / "medchem_hygiene_pass.csv"
PASS_SMI = HYG_DIR / "medchem_hygiene_pass.smi"
ANN_CSV  = HYG_DIR / "medchem_hygiene_annotated.csv"


def _load_from_csv(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    if "SMILES" not in df.columns:
        # sometimes a different column name is used
        smiles_col = "canonical_smiles" if "canonical_smiles" in df.columns else None
        if smiles_col is None:
            raise KeyError(f"No 'SMILES' column found in {csv_path}. Columns: {df.columns.tolist()}")
        df = df.rename(columns={smiles_col: "SMILES"})
    return df

def _load_from_smi(smi_path: Path) -> pd.DataFrame:
    smiles = [l.strip() for l in smi_path.read_text(encoding="utf-8").splitlines() if l.strip()]
    return pd.DataFrame({"SMILES": smiles})

df_in = None
input_source = None

if PREFER_PASS_CSV:
    if PASS_CSV.exists():
        df_in = _load_from_csv(PASS_CSV); input_source = "PASS_CSV"
    elif PASS_SMI.exists():
        df_in = _load_from_smi(PASS_SMI); input_source = "PASS_SMI"
    elif ANN_CSV.exists():
        ann = pd.read_csv(ANN_CSV)
        if "SMILES" not in ann.columns and "canonical_smiles" in ann.columns:
            ann = ann.rename(columns={"canonical_smiles": "SMILES"})
        df_in = ann[ann.get("MedChem_Pass", False) == True].copy()
        input_source = "ANN_CSV (filtered MedChem_Pass==True)"
    else:
        raise FileNotFoundError("No suitable hygiene outputs found.")
else:
    if PASS_SMI.exists():
        df_in = _load_from_smi(PASS_SMI); input_source = "PASS_SMI"
    elif PASS_CSV.exists():
        df_in = _load_from_csv(PASS_CSV); input_source = "PASS_CSV"
    elif ANN_CSV.exists():
        ann = pd.read_csv(ANN_CSV)
        if "SMILES" not in ann.columns and "canonical_smiles" in ann.columns:
            ann = ann.rename(columns={"canonical_smiles": "SMILES"})
        df_in = ann[ann.get("MedChem_Pass", False) == True].copy()
        input_source = "ANN_CSV (filtered MedChem_Pass==True)"
    else:
        raise FileNotFoundError("No suitable hygiene outputs found.")

df_in = df_in.dropna(subset=["SMILES"]).copy()

print(f"Input source = {input_source} | n = {len(df_in)}")
df_in.head(2)


Using hygiene run: /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/2-medchem_hygiene/mao-b_hygiene-20251225-094837
Input source = PASS_CSV | n = 327


,SMILES,MW,cLogP,RB,TPSA,HBD,HBA,Alert,MedChem_Pass,Why
0,Cc1ccc(NC(=O)c2ccc3c(cnn3C)c2)cc1Cl,299.761,3.78742,2,46.92,1,3,False,True,OK
1,O=C(Nc1cccc(Cl)c1)c1coc2sccc2c1=O,305.742,3.76020,2,59.31,1,4,False,True,OK


# **2) Create a Dedicated Output Folder for This ADMET Screening Run**

In [ ]:
# =========================
# Step 3: ADMET pass outputs
# =========================
run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
ADMET_DIR = ADMET_BASE / f"{target}_admet-{run_id}"
ADMET_DIR.mkdir(parents=True, exist_ok=True)
print("ADMET_DIR:", ADMET_DIR)

# Define Step 3 outputs
PRED_CSV   = ADMET_DIR / "step3_admet_preds.csv"         # raw predictions from ADMET model
ANN_CSV_3  = ADMET_DIR / "step3_admet_annotated.csv"     # predictions joined with medchem context (if available)
PASS_CSV_3 = ADMET_DIR / "step3_admet_pass.csv"          # rows passing ADMET gates
PASS_SMI_3 = ADMET_DIR / "step3_admet_pass.smi"          # SMILES of pass set
SUMMARY_3  = ADMET_DIR / "step3_summary.csv"             # counts/pass rates
README_3   = ADMET_DIR / "step3_README.txt"              # log of rules used


ADMET_DIR: /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/3-admet_pass/mao-b_admet-20251225-102420


# **3) Predict ADMET and Toxicity Properties for Each Molecule (can use ADMET-AI or custom code here)**

In [ ]:
"""

from admet_ai import ADMETModel

# 1) Build the model (loads the default prediction panel)
mdl = ADMETModel()

# 2) Predict on your SMILES
smiles_list = df_in["SMILES"].astype(str).tolist()
preds = mdl.predict(smiles_list)

# 3) Save predictions
preds.to_csv(PRED_CSV, index=False)
print("Saved raw ADMET-AI predictions →", PRED_CSV)

preds.head(2)

"""


'\n\nfrom admet_ai import ADMETModel\n\n# 1) Build the model (loads the default prediction panel)\nmdl = ADMETModel()\n\n# 2) Predict on your SMILES\nsmiles_list = df_in["SMILES"].astype(str).tolist()\npreds = mdl.predict(smiles_list)\n\n# 3) Save predictions\npreds.to_csv(PRED_CSV, index=False)\nprint("Saved raw ADMET-AI predictions →", PRED_CSV)\n\npreds.head(2)\n\n'

In [ ]:

USE_STUB = True  # set to False when using a real predictor

if not USE_STUB:
    # ---- REAL ADMET-AI EXAMPLE (uncomment after installing) ----
    !pip -q install admet-ai
    from admet_ai import ADMETModel
    mdl = ADMETModel()  # loads default panel
    preds = mdl.predict(df_in["SMILES"].astype(str).tolist())
    preds.to_csv(PRED_CSV, index=False)
    print("Saved raw predictions →", PRED_CSV)
    raise NotImplementedError("Set USE_STUB=False and add your predictor code.")
else:
    # ---- STUB: generates a structurally correct table that can gate on ----
    # Replace this with actual predictions when ready.
    import numpy as np
    rng = np.random.default_rng(42)
    n = len(df_in)
    preds = pd.DataFrame({
        "SMILES": df_in["SMILES"].astype(str).tolist(),
        "HIA": rng.choice(["High","Moderate","Low"], size=n, p=[0.6,0.3,0.1]),
        "Caco2": rng.choice(["High","Moderate","Low"], size=n, p=[0.4,0.4,0.2]),
        "logBB": rng.normal(loc=-0.4, scale=0.4, size=n).round(2),   # higher is better for BBB
        "hERG_risk": rng.choice(["Low","Medium","High"], size=n, p=[0.7,0.2,0.1]),
        "DILI_risk": rng.choice(["Low","Medium","High"], size=n, p=[0.8,0.15,0.05]),
        "CYP3A4_inhib": rng.choice(["No","Weak","Strong"], size=n, p=[0.7,0.2,0.1]),
        "Clearance": rng.choice(["Low","Moderate","High","Very High"], size=n, p=[0.25,0.45,0.25,0.05]),
    })
    preds.to_csv(PRED_CSV, index=False)
    print("Saved STUB predictions →", PRED_CSV)

preds.head(2)



Saved STUB predictions → /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/3-admet_pass/mao-b_admet-20251225-102420/step3_admet_preds.csv


,SMILES,HIA,Caco2,logBB,hERG_risk,DILI_risk,CYP3A4_inhib,Clearance
0,Cc1ccc(NC(=O)c2ccc3c(cnn3C)c2)cc1Cl,Moderate,Moderate,-1.37,Medium,Low,No,Moderate
1,O=C(Nc1cccc(Cl)c1)c1coc2sccc2c1=O,High,Moderate,0.24,Low,Low,No,Very High


# **4) Combine ADMET Predictions with the Existing Molecular Dataset**

In [ ]:
# --- Join predictions with input context (keeps pred_pIC50 if it exists in df_in) ---
if input_source in {"PASS_CSV", "ANN_CSV (filtered MedChem_Pass==True)"}:
    ann2 = df_in.merge(preds, on="SMILES", how="left")
else:
    # PASS_SMI path: df_in only has SMILES; just use preds table
    ann2 = preds.copy()

# SAVE ANNOTATED (this was missing)
ann2.to_csv(ANN_CSV_3, index=False)
print("Saved annotated CSV →", ANN_CSV_3)

# (sanity check)
print("pIC50 columns carried:", [c for c in ann2.columns if "pic50" in c.lower()])





Saved annotated CSV → /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/3-admet_pass/mao-b_admet-20251225-102420/step3_admet_annotated.csv
pIC50 columns carried: []


# **5) Apply CNS-Oriented ADMET Filtering Criteria**

In [ ]:
# ---- CNS gate policy (tweak as necessary) ----
# Rationale:
#  - HIA: "High" (keep good oral absorption)
#  - Caco-2: "Moderate/High" (permeability proxy)
#  - BBB proxy: logBB > -1.2  (loose threshold; adjust according to predictor's scale)
#  - hERG: exclude "High"
#  - DILI: exclude "High"
#  - CYP3A4_inhib: exclude "Strong" (policy decision; can relax to 'review')

def pass_admet(row) -> bool:
    if row.get("hERG_risk") == "High": return False
    if row.get("DILI_risk") == "High": return False
    if row.get("HIA") != "High": return False
    if row.get("Caco2") not in {"Moderate","High"}: return False
    if float(row.get("logBB", -999)) <= -1.2: return False
    if row.get("CYP3A4_inhib") == "Strong": return False
    # Optionally gate "Very High" clearance (context-dependent)
    # if row.get("Clearance") == "Very High": return False
    return True

ann2["ADMET_Pass"] = ann2.apply(pass_admet, axis=1)

admet_pass = ann2[ann2["ADMET_Pass"] == True].copy()
admet_pass.to_csv(PASS_CSV_3, index=False)
admet_pass[["SMILES"]].to_csv(PASS_SMI_3, index=False, header=False)

print(f"ADMET pass → {len(admet_pass)}/{len(ann2)}")
print("Saved pass CSV →", PASS_CSV_3)
print("Saved pass SMI →", PASS_SMI_3)


ADMET pass → 120/327
Saved pass CSV → /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/3-admet_pass/mao-b_admet-20251225-102420/step3_admet_pass.csv
Saved pass SMI → /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/3-admet_pass/mao-b_admet-20251225-102420/step3_admet_pass.smi


# **6) Save a concise summary + README**

In [ ]:
# Summary counts
summary = pd.DataFrame([{
    "target": target,
    "input_source": input_source,
    "n_input": len(df_in),
    "n_predicted": len(ann2),
    "n_admet_pass": len(admet_pass),
    "pass_rate_vs_pred": round(len(admet_pass)/max(1,len(ann2)), 3),
}])
summary.to_csv(SUMMARY_3, index=False)
summary


,target,input_source,n_input,n_predicted,n_admet_pass,pass_rate_vs_pred
0,mao-b,PASS_CSV,327,327,120,0.367


In [ ]:
# README for this run
readme_text = f"""\
STEP 2 — ADMET/TOX FILTER (CNS-oriented)
Run ID      : {run_id}
Target      : {target}
Input source: {input_source}
Hygiene dir : {HYG_DIR}

Files produced:
- step2_admet_preds.csv        : raw ADMET predictions per SMILES
- step2_admet_annotated.csv    : predictions joined with hygiene context (if available)
- step2_admet_pass.csv         : rows passing ADMET/Tox gates
- step2_admet_pass.smi         : SMILES of pass set (for tools)
- step2_summary.csv            : counts/pass rate

Gates used (default):
- HIA == High
- Caco2 in {{Moderate, High}}
- logBB > -1.2   (BBB proxy; tune to your model)
- hERG_risk != High
- DILI_risk != High
- CYP3A4_inhib != Strong
- (Optional) Clearance != Very High   # commented in code

Notes:
- model_used = "ADMET-AI (default panel)" if not USE_STUB else "STUB (synthetic)"
- Thresholds should be calibrated to the predictor scales and your project needs.
"""
README_3.write_text(readme_text, encoding="utf-8")
print("Saved README →", README_3)


Saved README → /content/drive/MyDrive/Colab Notebooks/ESOFT-MSC/thesis-project/GenAI/Filtering/3-admet_pass/mao-b_admet-20251225-102420/step3_README.txt


In [ ]:
"""
All results go to:
…/reinvent/<target>/samples/analyses/admet-<timestamp>/
    step2_admet_preds.csv
    step2_admet_annotated.csv
    step2_admet_pass.csv
    step2_admet_pass.smi
    step2_summary.csv
    step2_README.txt


"""

'\nAll results go to:\n…/reinvent/<target>/samples/analyses/admet-<timestamp>/\n    step2_admet_preds.csv\n    step2_admet_annotated.csv\n    step2_admet_pass.csv\n    step2_admet_pass.smi\n    step2_summary.csv\n    step2_README.txt\n\n\n'

In [ ]:
print(
    f"[Step2] Target={target} | Input={len(df_in)} | "
    f"Predicted={len(ann2)} | Passed ADMET={len(admet_pass)} "
    f"({len(admet_pass)/max(1,len(ann2)):.1%})"
)


[Step2] Target=mao-b | Input=327 | Predicted=327 | Passed ADMET=120 (36.7%)
